# PCL detection: LR scheduler experiment (linear vs cosine)

Based on `pcl_imbalance_weights_only.ipynb`. Same setup (RoBERTa-base, class-weighted CE, HTML stripping). This notebook compares **default (linear) LR scheduler** vs **cosine LR scheduler**. Effectiveness is judged by **F1 Pos on the dev dataset**.

In [1]:
import html
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
USE_CLASS_WEIGHTS = True  # class-weighted CE loss only

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Resolve project root regardless of whether notebook is run from project root or src/
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_BASE = PROJECT_ROOT / "models" / "roberta_pcl_lr_scheduler_experiment"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_PREFIX = "roberta_pcl_lr_scheduler_experiment"

MODEL_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

/vol/bitbucket/tq422/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Project root: /vol/bitbucket/tq422/NLP_cw


## Load data and build train/dev splits

In [2]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

# Split IDs (match organizer: exactly one row per ID from split files)
a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

train_df = a_train[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"] = dev_df["text"].fillna("").astype(str)
train_df["label_binary"] = train_df["label_binary"].fillna(0).astype(int)
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Split a validation set from training data; dev is held out and not used during training
VAL_RATIO = 0.1
train_df, val_df = train_test_split(
    train_df, test_size=VAL_RATIO, stratify=train_df["label_binary"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Dev size:   {len(dev_df)} (held out, not used during training)")
print("\nTrain label distribution:")
print(train_df["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Train size: 7537
Val size:   838
Dev size:   2094 (held out, not used during training)

Train label distribution:
label_binary
0    6822
1     715
Name: count, dtype: int64

Val label distribution:
label_binary
0    759
1     79
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


## Preprocess text: remove HTML markups

From data exploration: texts can contain HTML tags (e.g. `<h>`, `</p>`) that do not add useful signal. Strip tags and unescape HTML entities, then normalize whitespace.

In [3]:
def strip_html_and_clean(text: str) -> str:
    """Remove HTML tags, unescape entities, and normalize whitespace."""
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    # Remove HTML tags (pattern from data_exploration: <[^>]+>)
    no_tags = re.sub(r"<[^>]+>", "", text)
    # Unescape HTML entities (e.g. &amp; -> &, &lt; -> <)
    unescaped = html.unescape(no_tags)
    # Collapse multiple spaces and strip
    return re.sub(r"\s+", " ", unescaped).strip()


train_df["text"] = train_df["text"].apply(strip_html_and_clean)
val_df["text"] = val_df["text"].apply(strip_html_and_clean)
dev_df["text"] = dev_df["text"].apply(strip_html_and_clean)

# Sanity check: ensure no empty texts
train_df["text"] = train_df["text"].replace("", " ")
val_df["text"] = val_df["text"].replace("", " ")
dev_df["text"] = dev_df["text"].replace("", " ")
print("HTML preprocessing applied to train, val, and dev texts.")

HTML preprocessing applied to train, val, and dev texts.


In [4]:
# Class weights for loss only (inverse frequency; no sampling)
n0 = (train_df["label_binary"] == 0).sum()
n1 = (train_df["label_binary"] == 1).sum()
n = n0 + n1
w0 = n / (2 * n0)
w1 = n / (2 * n1)
class_weights = torch.tensor([w0, w1], dtype=torch.float32)

print(f"Class weights (0, 1): {class_weights.tolist()}")

Class weights (0, 1): [0.55240398645401, 5.270629405975342]


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(train_df["text"], train_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Training

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }


class ImbalanceTrainer(Trainer):
    """Trainer with class-weighted CE loss only (no sampling)."""

    def __init__(self, class_weights=None, use_class_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.use_class_weights = use_class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.use_class_weights and self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


print("compute_metrics and ImbalanceTrainer defined.")

compute_metrics and ImbalanceTrainer defined.


In [7]:
# --- Experiment 1: default (linear) LR scheduler ---
MODEL_DIR_LINEAR = MODEL_BASE / "lr_linear"
MODEL_DIR_LINEAR.mkdir(parents=True, exist_ok=True)

args_linear = TrainingArguments(
    output_dir=str(MODEL_DIR_LINEAR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)
# Default lr_scheduler_type is "linear" (no need to set explicitly)

model_linear = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer_linear = ImbalanceTrainer(
    model=model_linear,
    args=args_linear,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    class_weights=class_weights if USE_CLASS_WEIGHTS else None,
    use_class_weights=USE_CLASS_WEIGHTS,
)
trainer_linear.train()
trainer_linear.save_model(str(MODEL_DIR_LINEAR))
tokenizer.save_pretrained(str(MODEL_DIR_LINEAR))

dev_eval_linear = trainer_linear.evaluate(dev_dataset)
dev_f1_pos_linear = dev_eval_linear["eval_f1_pos"]
print(f"Experiment 1 (linear LR) — Dev F1 Pos: {dev_f1_pos_linear:.4f}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.520800,0.646343,0.595465,0.184466,0.962025,0.309572,0.511748
2,0.456200,0.472716,0.877088,0.409091,0.683544,0.511848,0.720771
3,0.326000,0.730150,0.890215,0.438095,0.582278,0.500000,0.719169
4,0.217600,0.969100,0.897375,0.469565,0.683544,0.556701,0.749336
5,0.101800,1.948096,0.916468,0.595745,0.354430,0.444444,0.699642
6,0.117200,2.016017,0.902148,0.476190,0.379747,0.422535,0.684540
7,0.058800,2.119765,0.917661,0.580645,0.455696,0.510638,0.732844


Experiment 1 (linear LR) — Dev F1 Pos: 0.6030


In [8]:
# --- Experiment 2: cosine LR scheduler ---
MODEL_DIR_COSINE = MODEL_BASE / "lr_cosine"
MODEL_DIR_COSINE.mkdir(parents=True, exist_ok=True)

args_cosine = TrainingArguments(
    output_dir=str(MODEL_DIR_COSINE / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

model_cosine = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer_cosine = ImbalanceTrainer(
    model=model_cosine,
    args=args_cosine,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    class_weights=class_weights if USE_CLASS_WEIGHTS else None,
    use_class_weights=USE_CLASS_WEIGHTS,
)
trainer_cosine.train()
trainer_cosine.save_model(str(MODEL_DIR_COSINE))
tokenizer.save_pretrained(str(MODEL_DIR_COSINE))

dev_eval_cosine = trainer_cosine.evaluate(dev_dataset)
dev_f1_pos_cosine = dev_eval_cosine["eval_f1_pos"]
print(f"Experiment 2 (cosine LR) — Dev F1 Pos: {dev_f1_pos_cosine:.4f}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.517600,0.591194,0.646778,0.202740,0.936709,0.333333,0.546537
2,0.432700,0.540690,0.856802,0.371069,0.746835,0.495798,0.706175
3,0.438100,1.041173,0.920048,0.593750,0.481013,0.531469,0.743882
4,0.257300,0.912929,0.883055,0.428571,0.721519,0.537736,0.735398
5,0.146100,1.494328,0.915274,0.557143,0.493671,0.523490,0.738497
6,0.098200,1.964994,0.917661,0.589286,0.417722,0.488889,0.722056
7,0.030700,2.020107,0.923628,0.622951,0.481013,0.542857,0.750595
8,0.003000,1.957953,0.922434,0.609375,0.493671,0.545455,0.751527
9,0.035000,2.000915,0.924821,0.625000,0.506329,0.559441,0.759172
10,0.003000,1.932525,0.922434,0.602941,0.518987,0.557823,0.757656


Experiment 2 (cosine LR) — Dev F1 Pos: 0.6059


In [9]:
# --- Compare: which LR scheduler is more effective? (Dev F1 Pos) ---
import json

results = [
    {"scheduler": "linear", "dev_f1_pos": dev_f1_pos_linear},
    {"scheduler": "cosine", "dev_f1_pos": dev_f1_pos_cosine},
]
comparison_df = pd.DataFrame(results)

print("Dev F1 Pos by LR scheduler:")
print(comparison_df.to_string(index=False))

best = max(results, key=lambda x: x["dev_f1_pos"])
print(f"\nMore effective: {best['scheduler'].upper()} LR (Dev F1 Pos = {best['dev_f1_pos']:.4f})")

summary = {
    "linear": {"dev_f1_pos": float(dev_f1_pos_linear), "model_dir": str(MODEL_DIR_LINEAR)},
    "cosine": {"dev_f1_pos": float(dev_f1_pos_cosine), "model_dir": str(MODEL_DIR_COSINE)},
    "best_scheduler": best["scheduler"],
}
with open(OUTPUT_DIR / f"{OUTPUT_PREFIX}_lr_scheduler_comparison.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved comparison to: {OUTPUT_DIR / f'{OUTPUT_PREFIX}_lr_scheduler_comparison.json'}")

Dev F1 Pos by LR scheduler:
scheduler  dev_f1_pos
   linear    0.602972
   cosine    0.605898

More effective: COSINE LR (Dev F1 Pos = 0.6059)

Saved comparison to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_lr_scheduler_experiment_lr_scheduler_comparison.json


In [10]:
# Both models were already saved in their experiment cells:
# - Linear LR: MODEL_DIR_LINEAR
# - Cosine LR: MODEL_DIR_COSINE
print(f"Linear LR model: {MODEL_DIR_LINEAR}")
print(f"Cosine LR model: {MODEL_DIR_COSINE}")

Linear LR model: /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_lr_scheduler_experiment/lr_linear
Cosine LR model: /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_lr_scheduler_experiment/lr_cosine


In [ ]:
# Optional: produce predictions for official test split using the better model (by dev F1 Pos)
best_trainer = trainer_linear if dev_f1_pos_linear >= dev_f1_pos_cosine else trainer_cosine
print(f"Using {'linear' if dev_f1_pos_linear >= dev_f1_pos_cosine else 'cosine'} LR model for test predictions.")

test_df = pd.read_csv(
    RAW_DATA_DIR / "task4_test.tsv",
    sep="\t",
    names=["id", "par_id", "keyword", "country_code", "text"],
)

test_df["text"] = test_df["text"].astype(str).apply(strip_html_and_clean)
test_df["text"] = test_df["text"].replace("", " ")

test_dataset = PCLDataset(
    texts=test_df["text"],
    labels=np.zeros(len(test_df), dtype=int),  # placeholder labels for dataset interface
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_pred_output = best_trainer.predict(test_dataset)
test_preds = np.argmax(test_pred_output.predictions, axis=-1)

test_txt_path = OUTPUT_DIR / "test.txt"
with open(test_txt_path, "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to: {test_txt_path}")
with open(PROJECT_ROOT / "test.txt", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to {PROJECT_ROOT / 'test.txt'} (root)")

Using cosine LR model for test predictions.


Saved test predictions to: /vol/bitbucket/tq422/NLP_cw/output/test.txt
Saved test predictions to /vol/bitbucket/tq422/NLP_cw/test.txt (root)


: 